In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
import sqlite3
import pandas as pd

DB_PATH = PROJECT_ROOT / "data" / "nifty100.db"

conn = sqlite3.connect(DB_PATH)

In [4]:
profit = pd.read_sql("SELECT * FROM profitandloss", conn)
balance = pd.read_sql("SELECT * FROM balancesheet", conn)
cash = pd.read_sql("SELECT * FROM cashflow", conn)
companies = pd.read_sql("SELECT * FROM companies", conn)

print(profit.shape)
print(balance.shape)
print(cash.shape)
print(companies.shape)

(1177, 15)
(1227, 13)
(1091, 7)
(92, 12)


In [5]:
profit = profit.drop_duplicates(subset=["company_id", "year"])
balance = balance.drop_duplicates(subset=["company_id", "year"])
cash = cash.drop_duplicates(subset=["company_id", "year"])

print(profit.shape)
print(balance.shape)
print(cash.shape)

(1164, 15)
(1058, 13)
(1057, 7)


In [6]:
df = (
    profit
    .merge(balance, on=["company_id", "year"], how="inner", suffixes=("", "_bs"))
    .merge(cash, on=["company_id", "year"], how="inner", suffixes=("", "_cf"))
)

print(df.shape)

(1044, 31)


In [7]:
duplicates = (
    df.groupby(["company_id", "year"])
      .size()
      .reset_index(name="count")
      .query("count > 1")
)

print(duplicates.shape)
duplicates.head()

(0, 3)


,company_id,year,count


In [8]:
df = (
    profit
    .merge(balance, on=["company_id", "year"], suffixes=("", "_bs"))
    .merge(cash, on=["company_id", "year"], suffixes=("", "_cf"))
)

df.shape

(1044, 31)

In [9]:
from src.analytics.ratios import (
    net_profit_margin,
    operating_profit_margin,
    roe,
    roce,
    roa,
    debt_to_equity,
    interest_coverage,
    asset_turnover,
)

from src.analytics.cashflow_kpis import (
    free_cash_flow,
)

In [10]:
df["net_profit_margin_pct"] = df.apply(
    lambda x: net_profit_margin(x["net_profit"], x["sales"]),
    axis=1
)

df["operating_profit_margin_pct"] = df.apply(
    lambda x: operating_profit_margin(
        x["operating_profit"],
        x["sales"]
    ),
    axis=1
)

df["return_on_equity_pct"] = df.apply(
    lambda x: roe(
        x["net_profit"],
        x["equity_capital"],
        x["reserves"]
    ),
    axis=1
)

df["debt_to_equity"] = df.apply(
    lambda x: debt_to_equity(
        x["borrowings"],
        x["equity_capital"],
        x["reserves"]
    ),
    axis=1
)

df["interest_coverage"] = df.apply(
    lambda x: interest_coverage(
        x["operating_profit"],
        x["other_income"],
        x["interest"]
    ),
    axis=1
)

df["asset_turnover"] = df.apply(
    lambda x: asset_turnover(
        x["sales"],
        x["total_assets"]
    ),
    axis=1
)

df["free_cash_flow_cr"] = df.apply(
    lambda x: free_cash_flow(
        x["operating_activity"],
        x["investing_activity"]
    ),
    axis=1
)

df["return_on_assets_pct"] = df.apply(
    lambda x: roa(
        x["net_profit"],
        x["total_assets"]
    ),
    axis=1
)

In [11]:
df[[
    "company_id",
    "year",
    "net_profit_margin_pct",
    "operating_profit_margin_pct",
    "return_on_equity_pct",
    "debt_to_equity",
    "interest_coverage",
    "asset_turnover",
    "free_cash_flow_cr",
    "return_on_assets_pct"
]].head()

,company_id,year,net_profit_margin_pct,operating_profit_margin_pct,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,free_cash_flow_cr,return_on_assets_pct
0,ABB,2012.0,8.771930,12.220206,22.411128,0.0,NaN,1.822492,42.0,15.986770
1,ABB,2014.0,8.699473,11.731107,25.126904,0.0,NaN,1.998244,11.0,17.383670
2,ABB,2015.0,10.004369,13.630406,24.439701,0.0,NaN,1.665939,28.0,16.666667
3,ABB,2016.0,9.755164,13.963275,21.338912,0.0,138.333333,1.617574,172.0,15.779703
4,ABB,2017.0,9.541853,13.709955,19.971161,0.0,227.500000,1.405131,152.0,13.407551


In [12]:
print(df.shape)

df.isna().sum().sort_values(ascending=False).head(20)

(1044, 39)


interest_coverage              43
operating_profit_margin_pct    13
opm_percentage                 13
operating_profit               12
eps                             4
dividend_payout                 4
financing_activity              2
operating_activity              2
net_cash_flow                   2
free_cash_flow_cr               2
investing_activity              2
return_on_assets_pct            1
net_profit_margin_pct           1
asset_turnover                  1
id                              0
sales                           0
expenses                        0
company_id                      0
year                            0
profit_before_tax               0
dtype: int64

In [13]:
df["return_on_capital_employed_pct"] = df.apply(
    lambda x: roce(
        x["operating_profit"],
        x["equity_capital"],
        x["reserves"],
        x["borrowings"]
    ),
    axis=1
)

In [30]:
ratio_df = df[
    [
        "company_id",
        "year",
        "net_profit_margin_pct",
        "operating_profit_margin_pct",
        "return_on_equity_pct",
        "return_on_assets_pct",
        "debt_to_equity",
        "interest_coverage",
        "asset_turnover",
        "free_cash_flow_cr",
        "return_on_capital_employed_pct",
    ]
].copy()

ratio_df.head()


,company_id,year,net_profit_margin_pct,operating_profit_margin_pct,return_on_equity_pct,return_on_assets_pct,debt_to_equity,interest_coverage,asset_turnover,free_cash_flow_cr,return_on_capital_employed_pct
0,ABB,2012.0,8.771930,12.220206,22.411128,15.986770,0.0,NaN,1.822492,42.0,31.221020
1,ABB,2014.0,8.699473,11.731107,25.126904,17.383670,0.0,NaN,1.998244,11.0,33.883249
2,ABB,2015.0,10.004369,13.630406,24.439701,16.666667,0.0,NaN,1.665939,28.0,33.297759
3,ABB,2016.0,9.755164,13.963275,21.338912,15.779703,0.0,138.333333,1.617574,172.0,30.543933
4,ABB,2017.0,9.541853,13.709955,19.971161,13.407551,0.0,227.500000,1.405131,152.0,28.695025


In [15]:
ratio_df.shape

(1044, 10)

In [16]:
ratio_df.to_sql(
    "financial_ratios",
    conn,
    if_exists="replace",
    index=False
)

print("financial_ratios table created successfully!")

financial_ratios table created successfully!


In [17]:
pd.read_sql(
    "SELECT COUNT(*) AS total_rows FROM financial_ratios",
    conn
)

,total_rows
0,1044


In [18]:
pd.read_sql(
    "SELECT * FROM financial_ratios LIMIT 10",
    conn
)

,company_id,year,net_profit_margin_pct,operating_profit_margin_pct,return_on_equity_pct,return_on_assets_pct,debt_to_equity,interest_coverage,asset_turnover,free_cash_flow_cr
0,ABB,2012.0,8.771930,12.220206,22.411128,15.986770,0.000000,NaN,1.822492,42.0
1,ABB,2014.0,8.699473,11.731107,25.126904,17.383670,0.000000,NaN,1.998244,11.0
2,ABB,2015.0,10.004369,13.630406,24.439701,16.666667,0.000000,NaN,1.665939,28.0
3,ABB,2016.0,9.755164,13.963275,21.338912,15.779703,0.000000,138.333333,1.617574,172.0
4,ABB,2017.0,9.541853,13.709955,19.971161,13.407551,0.000000,227.500000,1.405131,152.0
5,ABB,2018.0,12.158884,15.918739,23.685765,16.597682,0.000000,160.500000,1.365066,-62.0
6,ABB,2019.0,12.231585,16.444686,22.410359,15.300918,0.000000,359.000000,1.250935,242.0
7,ABB,2020.0,14.488151,18.494991,24.393254,16.718354,0.071987,96.777778,1.153933,225.0
8,ABB,2021.0,16.032483,21.392111,26.556495,17.994792,0.058801,55.722222,1.122396,655.0
9,ABB,2022.0,16.262976,22.023204,28.333333,18.915720,0.053901,61.315789,1.163116,552.0


In [19]:
ratio_df = df[
    [
        "company_id",
        "year",
        "net_profit_margin_pct",
        "operating_profit_margin_pct",
        "return_on_equity_pct",
        "return_on_assets_pct",
        "debt_to_equity",
        "interest_coverage",
        "asset_turnover",
        "free_cash_flow_cr",
    ]
].copy()

ratio_df.shape

(1044, 10)

In [20]:
print("Profit duplicates:")
print(
    profit.groupby(["company_id", "year"])
    .size()
    .reset_index(name="count")
    .query("count > 1")
    .head(20)
)

print("\nBalance duplicates:")
print(
    balance.groupby(["company_id", "year"])
    .size()
    .reset_index(name="count")
    .query("count > 1")
    .head(20)
)

print("\nCashflow duplicates:")
print(
    cash.groupby(["company_id", "year"])
    .size()
    .reset_index(name="count")
    .query("count > 1")
    .head(20)
)

Profit duplicates:
Empty DataFrame
Columns: [company_id, year, count]
Index: []

Balance duplicates:
Empty DataFrame
Columns: [company_id, year, count]
Index: []

Cashflow duplicates:
Empty DataFrame
Columns: [company_id, year, count]
Index: []


In [21]:
print("Profit rows:", len(profit))
print("Balance rows:", len(balance))
print("Cash rows:", len(cash))
print("Merged rows:", len(df))

Profit rows: 1164
Balance rows: 1058
Cash rows: 1057
Merged rows: 1044


In [22]:
print("Unique Profit:", profit[["company_id","year"]].drop_duplicates().shape)
print("Unique Balance:", balance[["company_id","year"]].drop_duplicates().shape)
print("Unique Cash:", cash[["company_id","year"]].drop_duplicates().shape)

Unique Profit: (1164, 2)
Unique Balance: (1058, 2)
Unique Cash: (1057, 2)


In [23]:
ratio_df["capex_cr"] = df["investing_activity"].abs()

ratio_df["earnings_per_share"] = df["eps"]

ratio_df["book_value_per_share"] = (
    df["equity_capital"] + df["reserves"]
) / df["equity_capital"]

ratio_df["dividend_payout_ratio_pct"] = df["dividend_payout"]

ratio_df["total_debt_cr"] = df["borrowings"]

ratio_df["cash_from_operations_cr"] = df["operating_activity"]

ratio_df.head()

,company_id,year,net_profit_margin_pct,operating_profit_margin_pct,return_on_equity_pct,return_on_assets_pct,debt_to_equity,interest_coverage,asset_turnover,free_cash_flow_cr,capex_cr,earnings_per_share,book_value_per_share,dividend_payout_ratio_pct,total_debt_cr,cash_from_operations_cr
0,ABB,2012.0,8.771930,12.220206,22.411128,15.986770,0.0,NaN,1.822492,42.0,59.0,68.0,30.809524,25.0,0.0,101.0
1,ABB,2014.0,8.699473,11.731107,25.126904,17.383670,0.0,NaN,1.998244,11.0,144.0,93.0,37.523810,25.0,0.0,155.0
2,ABB,2015.0,10.004369,13.630406,24.439701,16.666667,0.0,NaN,1.665939,28.0,187.0,108.0,44.619048,29.0,0.0,215.0
3,ABB,2016.0,9.755164,13.963275,21.338912,15.779703,0.0,138.333333,1.617574,172.0,77.0,120.0,56.904762,29.0,0.0,249.0
4,ABB,2017.0,9.541853,13.709955,19.971161,13.407551,0.0,227.500000,1.405131,152.0,155.0,130.0,66.047619,31.0,0.0,307.0


In [24]:
ratio_df.shape

(1044, 16)

In [25]:
ratio_df.to_sql(
    "financial_ratios",
    conn,
    if_exists="replace",
    index=False
)

print("financial_ratios updated successfully")

financial_ratios updated successfully


In [26]:
pd.read_sql(
    "SELECT COUNT(*) AS rows FROM financial_ratios",
    conn
)

,rows
0,1044


In [27]:
pd.read_sql(
    "PRAGMA table_info(financial_ratios);",
    conn
)

,cid,name,type,notnull,dflt_value,pk
0,0,company_id,TEXT,0,None,0
1,1,year,REAL,0,None,0
2,2,net_profit_margin_pct,REAL,0,None,0
3,3,operating_profit_margin_pct,REAL,0,None,0
4,4,return_on_equity_pct,REAL,0,None,0
5,5,return_on_assets_pct,REAL,0,None,0
6,6,debt_to_equity,REAL,0,None,0
7,7,interest_coverage,REAL,0,None,0
8,8,asset_turnover,REAL,0,None,0
9,9,free_cash_flow_cr,REAL,0,None,0


In [28]:
def quality_score(row):

    score = 0

    if pd.notna(row["return_on_equity_pct"]) and row["return_on_equity_pct"] >= 15:
        score += 1

    if pd.notna(row["debt_to_equity"]) and row["debt_to_equity"] <= 1:
        score += 1

    if pd.notna(row["interest_coverage"]) and row["interest_coverage"] >= 3:
        score += 1

    if pd.notna(row["net_profit_margin_pct"]) and row["net_profit_margin_pct"] >= 10:
        score += 1

    if pd.notna(row["asset_turnover"]) and row["asset_turnover"] >= 1:
        score += 1

    return score


ratio_df["composite_quality_score"] = ratio_df.apply(
    quality_score,
    axis=1
)

ratio_df.head()

,company_id,year,net_profit_margin_pct,operating_profit_margin_pct,return_on_equity_pct,return_on_assets_pct,debt_to_equity,interest_coverage,asset_turnover,free_cash_flow_cr,capex_cr,earnings_per_share,book_value_per_share,dividend_payout_ratio_pct,total_debt_cr,cash_from_operations_cr,composite_quality_score
0,ABB,2012.0,8.771930,12.220206,22.411128,15.986770,0.0,NaN,1.822492,42.0,59.0,68.0,30.809524,25.0,0.0,101.0,3
1,ABB,2014.0,8.699473,11.731107,25.126904,17.383670,0.0,NaN,1.998244,11.0,144.0,93.0,37.523810,25.0,0.0,155.0,3
2,ABB,2015.0,10.004369,13.630406,24.439701,16.666667,0.0,NaN,1.665939,28.0,187.0,108.0,44.619048,29.0,0.0,215.0,4
3,ABB,2016.0,9.755164,13.963275,21.338912,15.779703,0.0,138.333333,1.617574,172.0,77.0,120.0,56.904762,29.0,0.0,249.0,4
4,ABB,2017.0,9.541853,13.709955,19.971161,13.407551,0.0,227.500000,1.405131,152.0,155.0,130.0,66.047619,31.0,0.0,307.0,4


In [29]:
ratio_df.to_sql(
    "financial_ratios",
    conn,
    if_exists="replace",
    index=False
)

print("financial_ratios updated")

financial_ratios updated
